---
title: Load an export and check it
description: Load a Comment Scraper JSON export of a Reddit or Hacker News thread into pandas, check the reply tree, and handle hidden scores, deleted comments, the CSV version and Hacker News IDs.
---

Every chapter starts from the same file, a Reddit thread exported as "JSON, one row per comment". Before any analysis, this chapter checks that the reply tree in it holds together and goes through the details that quietly produce wrong numbers later: hidden scores, deleted comments, the CSV version of the same export and Hacker News threads.

In [ ]:
import pandas as pd

pd.set_option("display.width", 100)
pd.set_option("display.max_colwidth", 60)

## Load the file

An export holds a list of threads. A single export has one, a batch export has several. Each thread has a `meta` block and its rows. The nested JSON format stores replies inside their parents instead, so the loader first flattens it back into rows, in the same order: the post, then each comment followed by its replies.

```{literalinclude} threadtools.py
:start-at: def flatten_nested
:end-before: def check_tree
```

In [ ]:
from threadtools import load_export

records, meta = load_export("data/sample_thread.json")
meta[["thread_title", "task_state", "fetched",
      "known_unfetched_at_least", "reported_by_source"]].T

These are the three numbers the extension reports for every export, explained on the [help page](https://comment-scraper.com/docs#what-the-three-numbers-mean). The file holds 95 comments and the source reported 98. The status says no collapsed "more replies" link was left unopened, so the gap isn't comments the export skipped. I put it in the sample on purpose, because real threads have one too: the count a platform shows can include deleted comments and comments it no longer returns.

When the status is `partial`, treat every count you compute as a lower bound, and say in your write-up that medians and shares describe only the comments you have.

The nested export of the same thread loads to exactly the same table:

In [ ]:
nested, _ = load_export("data/sample_thread_nested.json")
nested.equals(records)

## Check the tree

```{literalinclude} threadtools.py
:start-at: def check_tree
:end-before: # ------
```

In [ ]:
from threadtools import check_tree

post, comments = check_tree(records)

No orphans and no depth mismatches: every reply points at a comment in the file, exactly one level up. On a partial export some replies will point at a parent that wasn't fetched. The extension marks those rows with `parent_missing`, and the check above counts them as orphans.

Deleted and removed comments stay in the file with an empty body. Their text is gone, but their place in the tree isn't:

In [ ]:
gone = comments[comments.comment_state != "normal"]
reply_counts = comments.parent_id.value_counts()
gone.assign(replies=gone.comment_id.map(reply_counts).fillna(0).astype(int))[
    ["comment_id", "depth", "comment_state", "replies"]
]

In [ ]:
assert len(gone) == 3
assert reply_counts["n128n6o"] == 1

The deleted top-level comment `n128n6o` still has a reply under it, and that reply has one of its own. Drop the deleted row and both lose their way back to the top of the thread. So filter deleted and removed comments out when you analyse text, and keep them when you analyse structure.

## Hidden scores are missing, not zero

A subreddit can hide new comments' scores for a while. When Reddit hides a score, the export leaves `score` empty and sets `score_hidden` to true. `load_export` reads the column as pandas' nullable integer type, so those two comments stay missing instead of turning into zeros or forcing the whole column to floats. It matters:

In [ ]:
scored = comments[comments.comment_state == "normal"]
level = scored.depth.clip(upper=3)  # 3 means "3 or deeper"
pd.DataFrame({
    "median, hidden skipped": scored.score.groupby(level).median(),
    "median, hidden as 0": scored.score.fillna(0).groupby(level).median(),
})

Filling the two hidden scores with zero pulls the top-level median from 88 to 84.5 and the depth-1 median from 26 to 25, and drops both comments to the bottom of any score ranking. Neither comment has a score of zero. Their scores just aren't known yet.

## The CSV version

The CSV export is meant for spreadsheets, and two things change on the way. The file starts with a byte order mark so that Excel reads it as UTF-8, and any text cell starting with `=`, `+`, `-` or `@` gets a leading apostrophe so a spreadsheet won't run it as a formula. A comment written as a Markdown list starts with a hyphen:

In [ ]:
ids = {"thread_id": str, "comment_id": str, "parent_id": str}
table = pd.read_csv("data/sample_thread.csv", encoding="utf-8-sig", dtype=ids,
                    keep_default_na=False, na_values=[""])

csv_comments = table[table.record_type == "comment"]
csv_body = csv_comments.set_index("comment_id").body.fillna("")
json_body = comments.set_index("comment_id").body
for comment_id in csv_body.index[csv_body != json_body]:
    print(f"[{comment_id}]  csv: {csv_body[comment_id][:28]!r}")
    print(f"{'':11}json: {json_body[comment_id][:28]!r}")

That's the only comment that differs between the two files. For analysis, read the JSON. If you have to read the CSV, pass the arguments above:

- `encoding="utf-8-sig"` drops the byte order mark. Without it, the first column is named `﻿schema_version`.
- `keep_default_na=False, na_values=[""]` treats only empty cells as missing. By default pandas also reads a cell that says `NA`, `null` or `None` as missing, and on a forum that's sometimes the whole comment.
- `dtype` keeps the IDs as text. Reddit IDs are letters and digits, so they survive anyway. Hacker News IDs are plain numbers, and the next section shows what pandas does to them.

## Hacker News threads

The same loader reads a Hacker News export:

In [ ]:
hn_records, hn_meta = load_export("data/sample_hn_thread.json")
hn_post, hn_comments = check_tree(hn_records)
hn_comments[["comment_id", "parent_id", "depth", "score", "is_op"]].head()

Three differences from Reddit:

- Hacker News doesn't show comment points, so `score` is empty for every comment. Anything in the next chapter that filters or sorts by score has nothing to work with on a Hacker News thread. Depth and OP still work.
- There's no subreddit, and the source's comment count is unknown, so `reported_by_source` is empty.
- IDs are numbers. In the JSON they're strings, which is how they should stay. Read the CSV without `dtype` and they don't:

In [ ]:
careless = pd.read_csv("data/sample_hn_thread.csv", encoding="utf-8-sig")
careless[["comment_id", "parent_id"]].head(3)

Both columns became floats, like `45120424.0`, because the post row has no comment ID and no parent, and pandas fills those gaps with `NaN`. Neither column matches the IDs in the JSON or in a permalink any more, and a join on them fails without an error. With `dtype=ids` both stay text.

The next chapter starts filtering. If you only want the comments in Python and you use LangChain, the [LangChain loader](https://comment-scraper.com/langchain) reads the same JSON files into one Document per comment.